# TP-1 — Conformité RGPD et anonymisation (portique avant ingestion)

**Cas d'usage :** prédiction du churn — éditeur SaaS B2B.

**Portée de ce notebook.** C'est l'étape qui précède la couche Bronze
(`01_ingestion_bronze.ipynb`) : avant de faire quoi que ce soit sur les données
brutes reçues du client, on vérifie la légalité et l'éthique de leur usage, on
recherche toute donnée directement identifiante, et on prépare une mesure de
pseudonymisation en défense en profondeur. **Aucune donnée métier n'est modifiée
ici** — ce notebook lit les fichiers sources en lecture seule et ne produit que des
artefacts de gouvernance (`data/rgpd/`).

**Remarque méthodologique.** Une analyse d'impact (AIPD) est en toute rigueur un
prérequis *avant* traitement, pas un bilan de fin de projet. Elle est reprise ici en
amont plutôt qu'en aval.

In [1]:
import hashlib
import json
import re
import secrets
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../Examen_cas d'usage candidat")
RGPD_DIR = Path("../data/rgpd")
RGPD_DIR.mkdir(parents=True, exist_ok=True)

CHECKED_AT = datetime.now(timezone.utc).isoformat()
LAYER_VERSION = "v1"

clients_raw = pd.read_csv(RAW_DIR / "churn_saas_complet.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
print("Fichier chargé en lecture seule :", RAW_DIR / "churn_saas_complet.csv")
print(f"{clients_raw.shape[0]} lignes x {clients_raw.shape[1]} colonnes")
print("Horodatage du contrôle :", CHECKED_AT)

Fichier chargé en lecture seule : ..\Examen_cas d'usage candidat\churn_saas_complet.csv
5035 lignes x 29 colonnes
Horodatage du contrôle : 2026-09-23T08:29:13.365156+00:00


## §1 — Checklist légale (base légale, finalité, minimisation)

Reprend les points soulevés avant toute ouverture des fichiers : quelle base légale
couvre ce traitement, la finalité (scoring churn) est-elle compatible avec la
finalité de collecte initiale, et quels articles du RGPD sont concrètement engagés.

In [2]:
checklist_legale = pd.DataFrame([
    {"point": "Base légale", "reference": "Art. 6",
     "statut": "À confirmer côté client",
     "commentaire": "Contrat / intérêt légitime probable -- à documenter formellement, pas supposé ici."},
    {"point": "Finalité et compatibilité d'usage", "reference": "Art. 5.1.b",
     "statut": "À confirmer côté client",
     "commentaire": "Le scoring churn doit être compatible avec la finalité de collecte initiale (facturation/support)."},
    {"point": "Minimisation", "reference": "Art. 5.1.c",
     "statut": "Vérifié dans ce notebook (§2)",
     "commentaire": "Aucune colonne directement identifiante détectée dans le schéma brut."},
    {"point": "Données sensibles", "reference": "Art. 9",
     "statut": "Vérifié dans ce notebook (§3)",
     "commentaire": "Recherche de motifs nominatifs / santé / opinions dans les champs texte libre."},
    {"point": "Décision automatisée", "reference": "Art. 22",
     "statut": "Conforme par conception",
     "commentaire": "Le score restera une recommandation -- validation humaine (CSM) obligatoire, jamais d'action automatique."},
    {"point": "Sous-traitance", "reference": "Art. 28",
     "statut": "À vérifier côté client",
     "commentaire": "DPA à formaliser si le traitement est hébergé/exécuté par un tiers pour le compte du client."},
    {"point": "Registre de traitement", "reference": "Art. 30",
     "statut": "À formaliser côté client",
     "commentaire": "Finalité, catégories de données, durées de conservation -- pas encore rédigé."},
])
checklist_legale

,point,reference,statut,commentaire
0,Base légale,Art. 6,À confirmer côté client,Contrat / intérêt légitime probable -- à docum...
1,Finalité et compatibilité d'usage,Art. 5.1.b,À confirmer côté client,Le scoring churn doit être compatible avec la ...
2,Minimisation,Art. 5.1.c,Vérifié dans ce notebook (§2),Aucune colonne directement identifiante détect...
3,Données sensibles,Art. 9,Vérifié dans ce notebook (§3),Recherche de motifs nominatifs / santé / opini...
4,Décision automatisée,Art. 22,Conforme par conception,Le score restera une recommandation -- validat...
5,Sous-traitance,Art. 28,À vérifier côté client,DPA à formaliser si le traitement est hébergé/...
6,Registre de traitement,Art. 30,À formaliser côté client,"Finalité, catégories de données, durées de con..."


## §2 — Scan des colonnes directement identifiantes

Recherche, sur les **noms de colonnes** du schéma brut, tout signal de données
directement identifiantes (nom, email, téléphone, adresse, IBAN, numéro de sécurité
sociale...).

In [3]:
PII_COLUMN_PATTERNS = re.compile(
    r"nom|prenom|pr\u00e9nom|email|e-mail|mail|telephone|t\u00e9l\u00e9phone|phone|"
    r"adresse|address|iban|rib|num_secu|ssn|siret|siren|naf|carte|card|password|mdp",
    re.IGNORECASE,
)

suspects = [c for c in clients_raw.columns if PII_COLUMN_PATTERNS.search(c)]
print("Colonnes potentiellement identifiantes détectées par nom :", suspects or "aucune")
print()
print("Schéma complet pour revue manuelle :")
for c in clients_raw.columns:
    print(" -", c)

Colonnes potentiellement identifiantes détectées par nom : aucune

Schéma complet pour revue manuelle :
 - client_id
 - date_souscription
 - jour_souscription
 - secteur
 - pays
 - taille_entreprise
 - plan
 - anciennete_mois
 - sieges_souscrits
 - utilisateurs_actifs
 - taux_adoption_pct
 - connexions_30j
 - heures_usage_30j
 - fonctionnalites_total
 - fonctionnalites_utilisees
 - nb_integrations
 - derniere_connexion_jours
 - tickets_support_90j
 - delai_reponse_support_h
 - csat
 - retards_paiement_12m
 - revenu_mensuel_recurrent_eur
 - couleur_theme_interface
 - code_datacenter
 - groupe_experimentation
 - commentaire_csm
 - sante_compte_fin_periode
 - valeur_vie_client_eur
 - churn


## §3 — Scan systématique des motifs nominatifs

Contrairement au premier passage (TP1), qui se limitait à un échantillon de
`commentaire_csm`, ce scan est **systématique** : il couvre toutes les colonnes
textuelles du fichier, sur l'intégralité des valeurs renseignées -- pas un
échantillon.

In [4]:
NAME_PATTERN = r"\b[A-ZÀ-Ü][a-zà-ü]+\s[A-ZÀ-Ü][a-zà-ü]+\b"
EMAIL_PATTERN = r"[\w.+-]+@[\w-]+\.[\w.-]+"
PHONE_PATTERN = r"(?:\+?\d[\s.-]?){8,12}\d"

text_columns = list(clients_raw.columns)  # dtype=str force à la lecture -- toutes les colonnes sont du texte

rows = []
for col in text_columns:
    # astype(object) : force le moteur regex Python (re), pas le moteur Arrow/RE2
    # utilisé par défaut par le dtype "str" de pandas 3.x, qui ne supporte pas
    # toutes les mêmes classes de caractères Unicode.
    values = clients_raw[col].astype(object).astype(str)
    non_empty = values[values.str.strip() != ""]
    n_names = non_empty.str.contains(NAME_PATTERN, regex=True).sum()
    n_emails = non_empty.str.contains(EMAIL_PATTERN, regex=True).sum()
    n_phones = non_empty.str.contains(PHONE_PATTERN, regex=True).sum()
    rows.append({
        "colonne": col,
        "valeurs_testees": len(non_empty),
        "motif_nom_prenom": int(n_names),
        "motif_email": int(n_emails),
        "motif_telephone": int(n_phones),
    })

scan_nominatif = pd.DataFrame(rows)
scan_nominatif

,colonne,valeurs_testees,motif_nom_prenom,motif_email,motif_telephone
0,client_id,5035,0,0,0
1,date_souscription,5035,0,0,0
2,jour_souscription,5035,0,0,0
3,secteur,4782,0,0,0
4,pays,4834,0,0,0
5,taille_entreprise,5035,0,0,0
6,plan,5035,0,0,0
7,anciennete_mois,5035,0,0,0
8,sieges_souscrits,5035,0,0,0
9,utilisateurs_actifs,5035,0,0,0


In [5]:
total_hits = scan_nominatif[["motif_nom_prenom", "motif_email", "motif_telephone"]].sum().sum()
print(f"Total de correspondances nominatives détectées, toutes colonnes texte confondues : {total_hits}")
if total_hits == 0:
    print("=> Aucun motif nominatif trouvé sur l'ensemble du fichier -- résultat plus fort que le "
          "contrôle par échantillon de TP1/TP2.")

Total de correspondances nominatives détectées, toutes colonnes texte confondues : 0
=> Aucun motif nominatif trouvé sur l'ensemble du fichier -- résultat plus fort que le contrôle par échantillon de TP1/TP2.


In [6]:
client_id_pattern_ok = clients_raw["client_id"].str.match(r"^CLI-\d+$").all()
print("Tous les client_id suivent le format CLI-<numéro> (pas de nom en clair) :", client_id_pattern_ok)

Tous les client_id suivent le format CLI-<numéro> (pas de nom en clair) : True


## §4 — Analyse d'impact (AIPD) — critères CNIL, évalués en amont

Les 4 critères qui déclenchent une obligation d'AIPD, appliqués ici *avant*
traitement plutôt qu'en fin de projet.

In [7]:
criteres_cnil = pd.DataFrame([
    {"critere": "Scoring/évaluation systématique", "reponse": "Oui",
     "commentaire": "Tous les comptes, à chaque cycle."},
    {"critere": "Décision produisant des effets significatifs", "reponse": "Non",
     "commentaire": "Recommandation, validation humaine obligatoire (art. 22)."},
    {"critere": "Personnes physiques directement ciblées", "reponse": "Non",
     "commentaire": "L'objet du scoring est le compte client (entreprise)."},
    {"critere": "Données sensibles à grande échelle", "reponse": "Non",
     "commentaire": "Aucune donnée art. 9 identifiée (§2-3)."},
])
n_criteres_positifs = (criteres_cnil["reponse"] == "Oui").sum()
print(criteres_cnil.to_string(index=False))
print()
print(f"Critères CNIL réunis : {n_criteres_positifs} / 4")
aipd_requise = n_criteres_positifs >= 2
print("AIPD obligatoire :", "Oui" if aipd_requise else "Non, en l'état -- à ré-évaluer si le périmètre change")

                                     critere reponse                                               commentaire
             Scoring/évaluation systématique     Oui                         Tous les comptes, à chaque cycle.
Décision produisant des effets significatifs     Non Recommandation, validation humaine obligatoire (art. 22).
     Personnes physiques directement ciblées     Non     L'objet du scoring est le compte client (entreprise).
          Données sensibles à grande échelle     Non                   Aucune donnée art. 9 identifiée (§2-3).

Critères CNIL réunis : 1 / 4
AIPD obligatoire : Non, en l'état -- à ré-évaluer si le périmètre change


## §5 — Pseudonymisation de précaution sur `client_id`

Le scan (§2-3) ne trouve aucune donnée directement identifiante : `client_id` est un
identifiant de compte, pas un nom, et aucun motif nominatif n'apparaît dans les
champs texte. La pseudonymisation n'est donc **pas indispensable** ici -- mais elle
reste une mesure de précaution peu coûteuse, recommandée dès le premier échange sur
ce projet : elle évite qu'un `client_id` circulant vers un système externe (export
CRM, TP9) permette un recoupement trivial avec la base source.

Principe : hachage salé, table de correspondance stockée **séparément**, avec un
accès distinct de celui des données de travail.

In [8]:
salt = secrets.token_hex(16)


def pseudonymize(client_id: str, salt: str) -> str:
    return hashlib.sha256((salt + client_id).encode("utf-8")).hexdigest()[:16]


keymap = pd.DataFrame({
    "client_id": clients_raw["client_id"],
    "client_key": clients_raw["client_id"].map(lambda cid: pseudonymize(cid, salt)),
})

assert keymap["client_key"].nunique() == keymap["client_id"].nunique(), "collision de hachage détectée"
print(f"Table de correspondance générée : {len(keymap)} identifiants pseudonymisés, 0 collision.")
keymap.head(3)

Table de correspondance générée : 5035 identifiants pseudonymisés, 0 collision.


,client_id,client_key
0,CLI-002447,a154af19aed837f8
1,CLI-004125,ba30d738ebb17279
2,CLI-000087,737b900e8c921fd8


In [9]:
keymap_path = RGPD_DIR / "keymap_client_id.parquet"
keymap.to_parquet(keymap_path, index=False)

secret_path = RGPD_DIR / "keymap_secret.json"
with open(secret_path, "w", encoding="utf-8") as f:
    json.dump({
        "salt": salt,
        "avertissement": (
            "Secret de pseudonymisation -- accès restreint. Ne jamais committer ce "
            "fichier avec la table de correspondance dans le même périmètre d'accès "
            "que les données de travail (Bronze/Silver/Gold)."
        ),
    }, f, ensure_ascii=False, indent=2)

print("Table de correspondance écrite :", keymap_path)
print("Secret de hachage écrit séparément :", secret_path)
print("=> client_key peut remplacer client_id dans les couches Silver/Gold si un usage "
      "externe (export CRM) le justifie -- décision à confirmer avec le client, non appliquée ici.")

Table de correspondance écrite : ..\data\rgpd\keymap_client_id.parquet
Secret de hachage écrit séparément : ..\data\rgpd\keymap_secret.json
=> client_key peut remplacer client_id dans les couches Silver/Gold si un usage externe (export CRM) le justifie -- décision à confirmer avec le client, non appliquée ici.


## §6 — Décision du portique RGPD (gate)

In [10]:
decision = {
    "version": LAYER_VERSION,
    "sha256_source_csv": hashlib.sha256((RAW_DIR / "churn_saas_complet.csv").read_bytes()).hexdigest(),
    "controle_effectue_le": CHECKED_AT,
    "fichier_controle": "churn_saas_complet.csv",
    "colonnes_identifiantes_detectees": suspects,
    "motifs_nominatifs_detectes": int(total_hits),
    "format_client_id_conforme": bool(client_id_pattern_ok),
    "criteres_cnil_reunis": int(n_criteres_positifs),
    "aipd_obligatoire": bool(aipd_requise),
    "pseudonymisation_client_id": {
        "appliquee_par_defaut": False,
        "disponible": True,
        "table_correspondance": str(keymap_path),
    },
    "decision": "GO -- ingestion Bronze autorisée" if (not suspects and total_hits == 0) else "BLOQUE -- revue manuelle requise",
    "reste_a_faire_cote_client": [
        "Base légale et finalité à documenter formellement (Art. 6, 5.1.b)",
        "Registre de traitement à rédiger (Art. 30)",
        "Contrat de sous-traitance à vérifier si hébergement tiers (Art. 28)",
    ],
}

manifest_path = RGPD_DIR / "rgpd_gate_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)

print(json.dumps(decision, ensure_ascii=False, indent=2))
print()
print("Manifeste de gate écrit :", manifest_path)

{
  "version": "v1",
  "sha256_source_csv": "2fc45e8c3c74d3ddcdbda162c5bb50184587201b46d877cc06742afd13063844",
  "controle_effectue_le": "2026-09-23T08:29:13.365156+00:00",
  "fichier_controle": "churn_saas_complet.csv",
  "colonnes_identifiantes_detectees": [],
  "motifs_nominatifs_detectes": 0,
  "format_client_id_conforme": true,
  "criteres_cnil_reunis": 1,
  "aipd_obligatoire": false,
  "pseudonymisation_client_id": {
    "appliquee_par_defaut": false,
    "disponible": true,
    "table_correspondance": "..\\data\\rgpd\\keymap_client_id.parquet"
  },
  "decision": "GO -- ingestion Bronze autorisée",
  "reste_a_faire_cote_client": [
    "Base légale et finalité à documenter formellement (Art. 6, 5.1.b)",
    "Registre de traitement à rédiger (Art. 30)",
    "Contrat de sous-traitance à vérifier si hébergement tiers (Art. 28)"
  ]
}

Manifeste de gate écrit : ..\data\rgpd\rgpd_gate_manifest.json


## Journal de bord — Synthèse TP-1

- **Checklist légale.** 3 points relèvent du client (base légale, finalité,
  registre, DPA) -- pas du périmètre technique de ce notebook.
- **Colonnes identifiantes.** Aucune détectée par scan de schéma.
- **Motifs nominatifs.** Scan systématique (pas un échantillon) sur toutes les
  colonnes texte -- 0 correspondance.
- **AIPD.** Critères CNIL évalués en amont, pas en fin de projet -- confirme la
  non-obligation en l'état.
- **Pseudonymisation.** Table de correspondance générée en précaution
  (`client_id` -> `client_key`), stockée séparément avec son secret de hachage --
  non appliquée par défaut, disponible si un usage externe le justifie.
- **Décision.** Gate GO -- l'ingestion Bronze (`01_ingestion_bronze.ipynb`) peut
  procéder sur les fichiers sources tels quels.
- **Prochaine étape.** `01_ingestion_bronze.ipynb`, puis couche Silver (nettoyage,
  TP2).